# Land–Ocean Contrast Processing for Overshooting Top (OT) Detections

This notebook processes OT detection files (2015–2025) to:
1. Regrid coarse OT detection data onto the BARRA-R2 land-fraction grid.
2. Split detections into **land** and **ocean** categories using a 50% land-fraction threshold.
3. Convert detection times from local solar to UTC-adjusted time.
4. Save per-year, per-type (land/ocean) results to CSV for downstream analysis.

Region of interest: whole of Australia (not just tropics), bounded by:
- Longitude: 109.895° – 158.035°
- Latitude: -44.9° – -9.565°


In [ ]:
import os
import glob
import xarray as xr
import pandas as pd
from datetime import datetime
import xesmf as xe
import intake
import numpy as np
#import proplot
import dask

from dask.distributed import Client 
dask.distributed.Client(threads_per_worker=1)

## Core Functions

This cell defines the three main functions used in the processing pipeline:

1. **`gridding`** — to regridd the coarsened OT data to BARRA grid
2. **`_preprocess_1 and _preprocess_2`** — functions for data preprocessing 
3. **`process_folders_land_ocean_contrast`** — orchestrates the full per-folder workflow: file discovery, batch loading, regridding, land/ocean masking, time conversion, and CSV export.

In [2]:
def gridding(ds_coarse, ds_out):

    """
    Regrid ds_coarse onto the grid of ds_out (subset to matching bounds).
    Uses nearest neighbour (source-to-destination).
    """

    # 1. Get spatial bounds from coarse dataset
    
    lat_min = round(ds_coarse.lat.min().item(), 4)
    lat_max = round(ds_coarse.lat.max().item(), 4)
    lon_min = round(ds_coarse.lon.min().item(), 4)
    lon_max = round(ds_coarse.lon.max().item(), 4)

    # 2. Subset output grid to same region
    ds_out_sub = ds_out.sel(
        lat=slice(lat_min, lat_max),
        lon=slice(lon_min, lon_max)
    )
    # 3. Ensure latitude ordering matches (descending)
    ds_out_sub = ds_out_sub.sortby('lat',ascending=False)

    # 4. Build regridder (nearest neighbour)
    gridder = xe.Regridder(ds_coarse, ds_out_sub, 
                           method='nearest_s2d',) # method = conservative is poorest form

    # 5. Apply regridding
    ds_gridded = gridder(ds_coarse)
    
    return ds_gridded

def _preprocess_1(ds):

    """
    preprocessing the file for part_1 folder
    """
    
    # 1. Drop unwanted variables
    # ds = ds.drop_vars(drop_variables_1)
    ds = ds.drop_vars([v for v in drop_variables_1 if v in ds])

    # 2. Subset to region of interest (note lat slice is reversed)
    ds = ds.sel(
        grid_lat=slice(lat_max, lat_min),
        grid_lon=slice(lon_min, lon_max)
    )

    # 3. Rename coordinates to standard names
    ds = ds.rename({
        "grid_lat": "lat",
        "grid_lon": "lon"
    })

    # 4. Convert to boolean mask (thresholding OT probability)
    ds = (ds['ot_probability'] >= 0.5)

    # 5. Coarsen grid (3x3) and take maximum within each block
    ds = ds.coarsen(
        lat=3,
        lon=3,
        boundary='trim',
        coord_func="first"
    ).max()

    return ds

def _preprocess_2(ds):

    """
    preprocessing the file second part of folder
    """
    
    # ds = ds.drop_vars(drop_variables_2)
    ds = ds.drop_vars([v for v in drop_variables_1 if v in ds])
    ds = ds.sel(grid_lat = slice(lat_max,lat_min), grid_lon = slice(lon_min,lon_max))
    ds = ds.rename({"grid_lat" : "lat", "grid_lon" : "lon"})
    ds = ds.rename({"maximum_ot_probability":"ot_probability"}) # extra for new files (after mid may) they named OT probability this time differently.
    ds = (ds['ot_probability'] >= 0.5)
    ds = ds.coarsen(lat = 3, lon = 3, boundary='trim',coord_func="first").max()

    return ds

def process_folders_land_ocean_contrast( folders,
                     data_path,
                     preprocess_fn,
                     land_select, #fraction of land
                     ds_out,
                     chunks = {'lat':-1, 'lon':-1},
                     exclude_files = None,    # list of specific files to skip
                     skip_empty_files = False,  # skip 0-byte files
                     skip_errors = False,       # skip corrupt/missing batches
                     batch_size = 30
                   ):

    """
    Process as set of year folders -applies masking, gridding, land/ocean split, and saves csv per year
    
    Args:
        folders        : list of folder names (Bedka's two part)
        data_path      : base path to NetCDF files
        preprocess_fn  : preprocessing function for xr.open_mfdataset
        chunks         : dask chunking strategy base don data
        exclude_files  : list of full file paths to exclude
        skip_empty_files : whether to skip empty files
        skip_errors    : whether to skip bad batches or raise
        batch_size     : nuber of files per batch to process

    """

    exclude_files = set(exclude_files or [])

    for folder in folders : 

        print(f"Processing folder : {folder}")

        land_batches = []
        ocean_batches = []
        skipped_files = []

        # --- File discovery ---

        all_files = sorted(glob.glob(f'{data_path}/{folder}/OCT_DETECT_AUS_Daily_*.nc'))

        #Apply exclusions

        files = [ f for f in all_files
                  if f not in exclude_files
                  and (not skip_empty_files or os.path.getsize(f) > 0)
                ]
        if not files :
            print(f" No valid files found for folder {folder}, skipping.")
            continue


        # - - - Batch Processing - - -

        for i in range(0, len(files), batch_size):

            batch = files[i : i+batch_size ]

            print (f"loading batch {i// batch_size + 1} : {len(batch)} files")

            try:

                ds_coarse = xr.open_mfdataset(
                    batch,
                    chunks = chunks,
                    parallel = True,
                    preprocess = preprocess_fn,
                )
            except (FileNotFoundError, OSError, ValueError) as e :
                if skip_errors:

                    print(f"[SKIPPED] Batch {i//batch_size +1} : {e}")
                    skipped_files.extend(batch)

                    continue
                else:
                    raise

            #Regrid
            # approximate way to regrid the land_frac to exisitng dataset (where we wanna use mask)
            # it has to be nearest and approximate, otherwise the numbers will be different
            ds_gridded = gridding(ds_coarse, ds_out)
    
            #land/Ocean masks
    
            land_frac = land_select.interp(
                lat = ds_gridded['lat'],
                lon = ds_gridded['lon'],
                method = 'nearest',
            )
    
            l_frac = xr.where(land_frac['sftlf'] > 50, 1, 0)
            o_frac = xr.where(land_frac['sftlf'] <= 50, 1, 0)
    
            # Land
            temp_land = (ds_gridded * l_frac).to_dataframe().dropna().reset_index()
            temp_land = temp_land[temp_land['ot_probability'] == 1]
            land_batches.append(temp_land)
    
            #Ocean
            temp_ocean = (ds_gridded * o_frac).to_dataframe().dropna().reset_index()
            temp_ocean = temp_ocean[temp_ocean['ot_probability'] == 1]
            ocean_batches.append(temp_ocean)
    
            # Validation
            value_counts = dict(zip(*np.unique(ds_gridded['ot_probability'], return_counts=True)))
            ot_count = value_counts.get(1, 0)
            print(f"  Validation: {ot_count == len(temp_land) + len(temp_ocean)} "
                  f"(OT={ot_count}, land={len(temp_land)}, ocean={len(temp_ocean)})")
    
    
            # Cleanup
            del temp_land, temp_ocean
            ds_coarse.close()
            ds_gridded.close()

        # --- Combine and save ---
        land_ot = pd.concat(land_batches, axis=0)
        ocean_ot = pd.concat(ocean_batches, axis=0)


        # UTC time conversion
        for df, label in [(land_ot, 'Land'), (ocean_ot, 'Ocean')]:
            df['type'] = label
            df['UTC_time'] = (
                pd.to_datetime(df['time']).dt.floor('h')
                - pd.to_timedelta(df['lon'] / 15, unit='h')
            ).dt.round('h')

        land_ot.to_csv(f"/g/data/if69/tp4064/Project_1/OT/dataframe_LAND_OT/{folder}_land_ot.csv")
        ocean_ot.to_csv(f"/g/data/if69/tp4064/Project_1/OT/dataframe_LAND_OT/{folder}_ocean_ot.csv")

        if skipped_files:
            print(f"\n  Total skipped files in {folder}: {len(skipped_files)}")
            for f in skipped_files:
                print(f"{f}")


    

# input for data processing such as folders, region of interest, land fraction file

In [6]:
# for 2015 to 2023

folder_1 = [2015]#, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 20231] #part 1
folder_2 = [20232]#, 2024, 2025] #part 2

# lat_max = -10 #start for upto -10
# lat_min = -20 #stop for upto -20
# lon_min = 120 #start for upto 120
# lon_max = 148 #stop for upto 148

# region of interest i.e whole Australia instead of just tropics
lon_min = 109.895
lon_max = 158.035
lat_min = -44.9 #threshold for OT data
lat_max =  -9.565

ds_1 = xr.open_dataset('/g/data/if69/tp4064/Project_1/OT/data/extracted/2015/OCT_DETECT_AUS_Daily_2015_200.nc', 
                            drop_variables =['ot_probability','grid_lon','grid_lat','time'] )
drop_variables_1 = list(ds_1.variables) # this is indirect way to extract variables for part 1 files


ds_2 = xr.open_dataset('/g/data/if69/tp4064/Project_1/OT/data/extracted/20232/OCT_DETECT_AUS_Daily_2023200.nc', 
                            drop_variables =['maximum_ot_probability','grid_lon','grid_lat','time'] )
drop_variables_2 = list(ds_2.variables) # this is indirect way to extract variables for part2 files



land = xr.open_dataset("/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/fx/sftlf/latest"+
"/sftlf_AUS-11_ERA5_historical_hres_BOM_BARRA-R2_v1.nc", drop_variables = ['crs'] ) 
# ustralian region land fraction file , used from barra because # next I will use parameter from barra
 
ds_out = land.copy(deep =True)  

# Rounding the grids can create issue since it uses method = nearest while selecting the , make sure the hit and trial must be done before finalising the 
#rounding.

land['lat'] = land['lat'].round(4)
land['lon'] = land['lon'].round(4)

land_select = land.sel(lat=slice(lat_min, lat_max), lon=slice(lon_min, lon_max))




# processing data and saving it

In [7]:
# Year group 1 — clean data
process_folders_land_ocean_contrast(
    folders=folder_1,
    data_path='/g/data/if69/tp4064/Project_1/OT/data/extracted',
    preprocess_fn=_preprocess_1,
    land_select = land_select,
    chunks={'lat': -1, 'lon': -1},
    skip_errors=False 
)
# Year group 2 — known corrupt/missing files
process_folders_land_ocean_contrast(
    folders=folder_2,
    data_path='/g/data/if69/tp4064/Project_1/OT/data/extracted',
    preprocess_fn=_preprocess_2,
    land_select = land_select,
    chunks='auto',
    exclude_files=['/g/data/if69/tp4064/Project_1/OT/data/extracted/2023/OCT_DETECT_AUS_Daily_2023188.nc'],
    skip_empty_files=True,
    skip_errors=True,
)

Processing folder : 2015
loading batch 1 : 30 files
  Validation: True (OT=9871, land=1738, ocean=8133)
loading batch 2 : 30 files
  Validation: True (OT=5961, land=578, ocean=5383)
loading batch 3 : 30 files
  Validation: True (OT=3687, land=128, ocean=3559)
loading batch 4 : 30 files
  Validation: True (OT=20741, land=15575, ocean=5166)
loading batch 5 : 30 files
  Validation: True (OT=24714, land=18063, ocean=6651)
loading batch 6 : 29 files
  Validation: True (OT=39516, land=17925, ocean=21591)
Processing folder : 20232
loading batch 1 : 30 files
  Validation: True (OT=6495, land=904, ocean=5591)
loading batch 2 : 30 files
  Validation: True (OT=6482, land=1067, ocean=5415)
loading batch 3 : 30 files
  Validation: True (OT=9535, land=1260, ocean=8275)
loading batch 4 : 30 files
  Validation: True (OT=5934, land=765, ocean=5169)
loading batch 5 : 30 files
  Validation: True (OT=28152, land=21786, ocean=6366)
loading batch 6 : 30 files
  Validation: True (OT=58815, land=28504, ocean=

# collecting all pandas dataframe together in single file for whole land and ocean.

In [ ]:
# collecting all pandas dataframe together in single file for whole land and ocean.

import glob
import os

# This is for Ocean , similarly it can be done for land
files = glob.glob("/g/data/if69/tp4064/Project_1/OT/dataframe_OCEAN_OT/*.csv")

# selecting the period needed for the analysis
files = [
    f for f in files
    if not os.path.basename(f).startswith(("2015", "2025"))
]


dfs = [pd.read_csv(f) for f in sorted(files)]
final_df = pd.concat(dfs, ignore_index = True)

# preprocessing the data before saving
final_df['UTC_time'] = pd.to_datetime(final_df['UTC_time'])
final_df = final_df[final_df['UTC_time'].dt.year != 2015].reset_index()
final_df= final_df.drop(["time","Unnamed: 0","index"], axis = 1)

# Saving the file
final_df.to_feather("/g/data/if69/tp4064/Project_1/OT/dataframe_OCEAN_OT/whole_Australia_OCEAN_OT_2016_2024.feather")
